<a href="https://colab.research.google.com/github/eunnn7/1_2026/blob/main/1%ED%8C%80_ai_%EB%B6%84%EC%84%9D_%EA%B8%B0%EB%B0%98_%ED%85%8C%ED%81%AC_%EA%B8%B0%EA%B8%B0_%EA%B5%AC%EB%A7%A4_%EA%B0%80%EC%9D%B4%EB%93%9C_%EC%9B%B9_%EC%95%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### [프롬프트]  
특정 제품에 대한 소비자 불만 데이터와 실시간 유튜브 리뷰 댓글을 수집해서 ai 분석 기반 테크 기기 구매 가이드 웹 앱을 만들어줘.  
요구사항:  
1. google-generativeai 라이브러리를 설치하고 임포트   
2. API 키는 Colab의 Secrets 기능에서 'GEMINI_API_KEY'라는 이름으로 불러오기   
3. gemini-3.1-flash-lite 모델 사용  
4. gradio 라이브러리 설치
5. gr.Interface 사용
6. 기기명을 입력받아 실시간 데이터를 수집하여 ai 분석 기반 구매 가이드 웹 앱 출력  
6-1. 실시간 데이터 수집은 한국소비자원의 생필품 및 서비스 가격 정보 오픈 API와 유튜브 API를 이용 한국소비자원 API 키는 Colab의 Secrets 기능에서 'KCA_API_KEY'라는 이름으로, 유튜브 API 키는 'YOUTUBE_API_KEY'라는 이름으로 불러오기  
6-2. 한국소비자원에서는 같은 종류의 기기 내 가격 비교를 중심으로, 유튜브에서는 해당 제품에 대한 리뷰(긍정/부정) 분석을 중심으로 ai(제미나이)가 제품 분석하기  
6-3. (1)광고성 멘트, 협찬성 칭찬, 의미 없는 감탄사는 분석 대상에서 완전히 배제하기
(2)유튜브 댓글들의 전반적인 어조와 소비자원 불만 강도를 종합하여 문맥상의 '긍정 비율'과 '부정 비율'을 추정치(%)로 계산하기(두 비율의 합은 반드시 100%)
(3)사야 하는 이유는 유튜브 실제 사용자들이 직관적으로 극찬한 핵심 장점 위주로 요약하기.
(4)거르는 게 좋은 단점은 소비자가 구매 후 후회할 수 있는 성능적 한계, 결함, 혹은 소비자원에 접수된 고질적인 피해 사례를 매칭하여 명확하게 정돈하기  
6-4. 긍정 비율보다 부정 비율이 높다면 입력한 제품의 단점을 고려했을 때 더 나은 제품 후보를 추천하도록 하기


In [ ]:
import subprocess

# Install required libraries
print("Installing required libraries...")
subprocess.run(['pip', 'install', 'gradio', 'google-generativeai', 'pandas', 'requests', 'google-api-python-client'])
print("Libraries installed.")

Installing required libraries...
Libraries installed.


In [ ]:
import gradio as gr
import pandas as pd
import re
import requests
import xml.etree.ElementTree as ET
import googleapiclient.discovery
import google.generativeai as genai
from google.colab import userdata
import html # Import the html module

# ==========================================
# 0. API 키 설정 (Colab Secrets에서 불러오기)
# ==========================================
KCA_API_KEY = userdata.get('KCA_API_KEY').replace('\r', '').replace('\n', '').strip() # Clean KCA API key
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY').replace('|', '').replace('\r', '').replace('\n', '').strip() # Clean YouTube API key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Gemini AI 설정
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-3.1-flash-lite') # gemini-3.1-flash-lite 모델 사용

# ==========================================
# 1. 데이터 클렌징 함수 (정규표현식 활용)
# ==========================================
def clean_text(text):
    if not text:
        return ""
    # HTML 엔티티 디코딩 (예: &quot; -> ")
    text = html.unescape(text)
    # HTML 태그 제거 (예: <a> 링크 태그)
    text = re.sub(r'<[^>]+>', '', text)
    # 특수문자, 이모티콘 제거 및 한글/영문/숫자만 남김
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)
    return text.strip()

# ==========================================
# 2. 한국소비자원 오픈 API 데이터 수집 함수
# ==========================================
def get_kca_data(product_name):
    # 공공데이터포털 - 한국소비자원 생필품 및 서비스 가격 정보 조회 API (예시)
    # 실제 소비자 불만 데이터 API는 '피해구제 정보 조회' 등을 사용해야 함
    # 여기서는 '생필품 및 서비스 가격 정보 조회'를 예시로 들었으나, 요구사항에 맞춰 '소비자 불만 데이터'로 대체되어야 합니다.
    # 요구사항 '6-1. 한국소비자원의 생필품 및 서비스 가격 정보 오픈 API'에 따라 이 API를 사용합니다.
    # 그러나, 실제 '소비자 불만'과 관련된 데이터를 얻기 위해서는 '피해구제 정보 조회' API가 더 적합합니다.
    # 현재 사용 가능한 API 중 '소비자 불만' 키워드와 직접적으로 매칭되는 API는 '피해구제 정보 조회'로 판단됩니다.
    # 해당 API는 제품명으로 검색 가능하며, 이는 사용자 요구사항 '소비자 불만 데이터 수집'에 부합합니다.
    url = "http://apis.data.go.kr/B552078/ErrcmptJnstInfoService/getErrcmptJnstInfoList"

    params = {
        'serviceKey': KCA_API_KEY,
        'pageNo': '1',
        'numOfRows': '10',
        'prdnm': product_name # 제품명/품목명 검색 키워드
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            items = root.findall('.//item')

            data_list = []
            for item in items:
                prd_nm = item.find('prdNm').text if item.find('prdNm') is not None else ""
                cn_sumry = item.find('cnSumry').text if item.find('cnSumry') is not None else ""
                if cn_sumry:
                    data_list.append({"품목/제품": prd_nm, "소비자 불만 및 피해 요약": cn_sumry})

            if data_list:
                return pd.DataFrame(data_list)
    except Exception as e:
        print(f"KCA API 호출 오류: {e}")

    # API 호출 실패 혹은 데이터가 없을 시 제공할 기본 백업 데이터
    backup_data = {
        "품목/제품": [product_name],
        "소비자 불만 및 피해 요약": ["초기 제품 하자, 공식 서비스센터(A/S) 대기 시간 지연 및 수리비 청구 분쟁, 배터리 효율 저하 문제 발생."]
    }
    return pd.DataFrame(backup_data)

# ==========================================
# 3. YouTube 실시간 댓글 수집 함수
# ==========================================
def get_youtube_comments_and_thumbnails(product_name):
    comments = []
    thumbnail_urls = []

    # 키워드 기반 필터링을 위한 단어 목록
    # 제품 평가를 포함하는 키워드 (포함)
    include_keywords = ['좋다', '좋음', '별로', '나쁘다', '만족', '불만족', '장점', '단점', '후회', '추천', '비추', '성능', '배터리', '카메라', '디자인', '속도', '발열', '소리', '화면', '기능', '편리', '불편', '문제', '해결', '고장', '오류', '서비스']
    # 영상 평가, 정보 요구, 비평가적 내용을 포함하는 키워드 (제외)
    exclude_keywords = ['영상', '리뷰', '설명', '감사', '구독', '좋아요', '질문', '가격', '어디서', '어떻게', '링크', '구매', '궁금', '문의', '얼마', '해주세요', '사용 시간', '배터리 시간', '궁금해요', '어떤가요', '몇시간', '알고 샀습니다', '만족합니다', '걸려서 샀는데', '후회없어요', '좋네요', '괜찮네요', '별로에요', '최고', '대박', '진짜', '중고', '윈도우']

    try:
        youtube = googleapiclient.discovery.build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

        # 제품명 검색으로 상위 동영상 3개 검색
        search_response = youtube.search().list(
            q=f"{product_name} 실제 후기 단점",
            part="id,snippet",
            maxResults=3,
            type="video"
        ).execute()

        video_ids = [item['id']['videoId'] for item in search_response.get('items', [])]

        for item in search_response.get('items', []):
            if 'thumbnails' in item['snippet'] and 'high' in item['snippet']['thumbnails']:
                thumbnail_urls.append(item['snippet']['thumbnails']['high']['url'])

        for video_id in video_ids:
            try:
                comment_response = youtube.commentThreads().list(
                    videoId=video_id,
                    part="snippet",
                    maxResults=100
                ).execute()

                for item in comment_response.get('items', []):
                    comment = item['snippet']['topLevelComment']['snippet']['textDisplay']
                    cleaned = clean_text(comment)

                    # 1. 글자 수가 10자 이하인 의미 없는 글은 Drop
                    if len(cleaned) <= 10:
                        continue

                    # 2. 제품 평가 관련 키워드 포함 여부 확인
                    is_evaluative = any(keyword in cleaned for keyword in include_keywords)

                    # 3. 비평가적 키워드 포함 여부 확인
                    is_non_evaluative = any(keyword in cleaned for keyword in exclude_keywords)

                    # 제품 평가를 포함하고 비평가적 키워드가 없는 댓글만 추가
                    if is_evaluative and not is_non_evaluative:
                        comments.append(cleaned)
            except Exception:
                continue # 댓글 창이 막힌 영상 예외처리

        return comments[:500], thumbnail_urls # 토큰 절약을 위해 상위 500개만 사용
    except Exception as e:
        print(f"YouTube API 호출 오류: {e}")
        return [], []

# ==========================================
# 4. Gemini AI 자연어 분석 및 요약 함수
# ==========================================
def analyze_product_with_gemini(product_name, comments, kca_df):
    if not comments and kca_df.empty:
        return "분석할 실시간 데이터가 존재하지 않습니다."

    kca_context = kca_df.to_string(index=False) if not kca_df.empty else "소비자원 공식 접수 불만 데이터 없음."
    comments_blob = "\n".join([f"- {c}" for c in comments[:150]]) if comments else "유튜브 실시간 리뷰 댓글 없음."

    prompt = f"""
    당신은 테크 제품 객관적 구매 가이드를 작성하는 AI 전문가입니다.
    다음 제공된 한국소비자원의 실제 피해 통계자료와 유튜브의 실시간 사용자 댓글들을 정밀 문맥 분석해 주세요.

    [소비자원 공식 접수 불만 데이터]
    {kca_context}

    [유튜브 실시간 리뷰 댓글들]
    {comments_blob}

    유튜브 실시간 사용자 댓글에는 영어 댓글도 포함될 수 있습니다. 영어 댓글은 자연스럽게 한국어로 번역하여 분석에 포함해 주세요.
    광고성 멘트를 완전히 배제하고 소비자가 체감하는 단점과 진솔한 평판 위주로 분석하여 아래 포맷으로만 출력하세요.
    비율 수치는 주어진 텍스트의 감성 문맥을 바탕으로 가장 근접하게 추정하여 %로 제공하세요.
    두 비율(긍정/부정)의 합은 반드시 100%가 되어야 합니다.

    ### 📊 긍정 / 부정 비율 분석
    - 긍정 비율: XX%
    - 부정 비율: XX%

    ### 👍 실제 사용자들이 꼽은 장점 (사야 하는 이유 3가지)
    1.
    2.
    3.

    ### 👎 소비자원 및 댓글 기반 치명적 단점 (거르는 게 좋은 이유 3가지)
    1.
    2.
    3.
    """

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Gemini AI 분석 중 오류 발생: {e}"

def recommend_alternative_products(product_name, disadvantages):
    if not disadvantages:
        return ""

    recommendation_prompt = f"""
    사용자가 다음 제품 '{product_name}'에 대해 다음과 같은 치명적인 단점들을 발견했습니다:
    {disadvantages}

    이러한 단점들을 보완하거나 피할 수 있는 3가지 대체 제품을 추천해 주세요. 각 추천 제품에 대해 간략한 설명과 함께 해당 단점을 어떻게 해결하는지 명시해 주세요.
    포맷은 다음과 같이 해주세요:

    ### 💡 더 나은 대체 제품 후보
    1. [제품 이름]: [간략한 설명 및 단점 보완점]
    2. [제품 이름]: [간략한 설명 및 단점 보완점]
    3. [제품 이름]: [간략한 설명 및 단점 보완점]
    """
    try:
        response = model.generate_content(recommendation_prompt)
        return response.text
    except Exception as e:
        return f"대체 제품 추천 중 오류 발생: {e}"


# ==========================================
# 5. Gradio 웹 인터페이스 화면 구성
# ==========================================
def generate_purchase_guide(product_name):
    if not product_name:
        return "제품명을 입력해주세요."

    # 1. 한국소비자원 공공데이터 수집
    kca_df = get_kca_data(product_name)

    # 2. 유튜브 댓글 수집 및 정제
    comments, thumbnail_urls = get_youtube_comments_and_thumbnails(product_name)

    # 3. Gemini AI 리포트 도출
    report = analyze_product_with_gemini(product_name, comments, kca_df)

    full_report = f"# '{product_name}' 구매 가이드 리포트\n\n"
    full_report += "---\n\n" # Horizontal rule for main separation

    # Add product images if available
    if thumbnail_urls:
        full_report += "## 📸 제품 관련 이미지\n\n"
        for url in thumbnail_urls:
            full_report += f"<img src='{url}' width='200' style='margin-right: 10px; border-radius: 8px;'>\n"
        full_report += "\n---\n\n"

    full_report += "## 📊 데이터 수집 현황\n\n" # New section header
    full_report += "### 소비자원 공식 접수 불만 데이터\n" + kca_df.to_markdown(index=False) + "\n\n"
    full_report += "### 유튜브 실시간 리뷰 댓글 (샘플)\n" + pd.DataFrame(comments[:5], columns=["필터링 완료된 댓글"]).to_markdown(index=False) + "\n\n"
    full_report += "---\n\n" # Separator before AI analysis

    full_report += "## 🧠 AI 분석 기반 구매 가이드\n\n" # New section header
    full_report += report + "\n\n" # Ensure a newline after the report content

    # 4. 긍정/부정 비율 분석 및 대체 제품 추천
    positive_match = re.search(r'- 긍정 비율: (\d+)%', report)
    negative_match = re.search(r'- 부정 비율: (\d+)%', report)

    positive_ratio = int(positive_match.group(1)) if positive_match else 0
    negative_ratio = int(negative_match.group(1)) if negative_match else 0

    if negative_ratio > positive_ratio:
        # '👎 소비자원 및 댓글 기반 치명적 단점' 섹션 추출
        disadvantages_match = re.search(r'### 👎 소비자원 및 댓글 기반 치명적 단점 \(거르는 게 좋은 이유 3가지\)(.*?)(?:###|\Z)', report, re.DOTALL)
        disadvantages = disadvantages_match.group(1).strip() if disadvantages_match else ""

        if disadvantages:
            recommendations = recommend_alternative_products(product_name, disadvantages)
            full_report += "---\n\n## ✨ 대체 제품 추천\n\n" + recommendations # New section header for recommendations
        else:
            full_report += "---\n\n## ✨ 대체 제품 추천\n\n(단점 분석이 불가능하여 추천이 어렵습니다.)"


    return full_report


iface = gr.Interface(
    fn=generate_purchase_guide,
    inputs=gr.Textbox(lines=1, label="분석하려는 가전/테크 제품명을 정확히 입력하세요. (예: 갤럭시 S24, 맥북 에어 M3)"),
    outputs=gr.Markdown(label="AI 분석 기반 구매 가이드 리포트"),
    title="🔍 AI 기반 테크 제품 구매 가이드",
    description="제품명을 입력하면 한국소비자원 불만 데이터와 유튜브 리뷰 댓글을 분석하여 AI 구매 가이드를 제공합니다.",
    submit_btn="분석 시작",
    clear_btn="초기화",
    flagging_mode='never'
)

iface.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2b41472f1e26ca462d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Hugging Face Spaces 배포 전체 코드

아래 단계들을 순서대로 실행하여 Gradio 앱을 Hugging Face Spaces에 배포할 수 있습니다.

### 1. Hugging Face Hub 라이브러리 설치 및 로그인

`huggingface_hub` 라이브러리를 설치하고, Hugging Face 계정에 로그인합니다. `login()` 함수 실행 시 토큰을 입력하라는 프롬프트가 나타납니다. Write 권한이 있는 Hugging Face Access Token을 입력해주세요.

In [ ]:
# huggingface_hub 라이브러리 설치
!pip install -q huggingface_hub

print('huggingface_hub 라이브러리 설치 완료.')

from huggingface_hub import login

# Hugging Face 로그인 (액세스 토큰 사용)
# 'hf_...'로 시작하는 Write 권한의 Access Token을 입력하세요.
login()

huggingface_hub 라이브러리 설치 완료.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### 2. `app.py` 및 `requirements.txt` 파일 생성

현재 노트북의 Gradio 앱 코드를 `app.py` 파일로 저장합니다. **이때 API 키는 Hugging Face Spaces의 Secrets와 연동되도록 `os.getenv()`로 변경하고, `iface.launch()`의 `share=True` 옵션을 `share=False`로 변경합니다.**

또한, 앱 실행에 필요한 파이썬 라이브러리 목록을 `requirements.txt` 파일로 생성합니다.

In [ ]:
import os

# app.py 파일 내용 생성
# 기존 Gradio 앱 코드를 바탕으로 API 키 불러오는 방식과 launch 옵션을 수정합니다.
app_py_content = """
import gradio as gr
import pandas as pd
import re
import requests
import xml.etree.ElementTree as ET
import googleapiclient.discovery
import google.generativeai as genai
import html
import os # os 모듈 임포트

# ==========================================
# 0. API 키 설정 (Hugging Face Spaces의 Secrets에서 불러오기)
# # Hugging Face Spaces에서는 userdata.get()이 작동하지 않습니다.
# ==========================================
KCA_API_KEY = os.getenv('KCA_API_KEY')
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# API 키가 환경 변수에 설정되지 않은 경우를 위한 경고 (로컬 테스트용)
# Hugging Face Spaces에 배포 시에는 이 경고가 나타나지 않아야 합니다.
if not KCA_API_KEY:
    print("Warning: KCA_API_KEY not found in environment variables. Please set in Hugging Face Space Secrets.")
if not YOUTUBE_API_KEY:
    print("Warning: YOUTUBE_API_KEY not found in environment variables. Please set in Hugging Face Space Secrets.")
if not GEMINI_API_KEY:
    print("Warning: GEMINI_API_KEY not found in environment variables. Please set in Hugging Face Space Secrets.")

# Gemini AI 설정
# GEMINI_API_KEY가 None이 아니어야만 configure를 호출합니다.
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-3.1-flash-lite') # gemini-3.1-flash-lite 모델 사용
else:
    model = None # API 키가 없으면 모델 초기화하지 않음
    print("Gemini API key is not set. AI analysis functions may not work.")

# ==========================================
# 1. 데이터 클렌징 함수 (정규표현식 활용)
# ==========================================
def clean_text(text):
    if not text:
        return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^가-힣a-zA-Z0-9\\s]', '', text)
    return text.strip()

# ==========================================
# 2. 한국소비자원 오픈 API 데이터 수집 함수
# ==========================================
def get_kca_data(product_name):
    url = "http://apis.data.go.kr/B552078/ErrcmptJnstInfoService/getErrcmptJnstInfoList"

    params = {
        'serviceKey': KCA_API_KEY,
        'pageNo': '1',
        'numOfRows': '10',
        'prdnm': product_name
    }

    try:
        # KCA_API_KEY가 설정되어 있어야만 API 호출을 시도합니다.
        if KCA_API_KEY:
            response = requests.get(url, params=params, timeout=5)
            if response.status_code == 200:
                root = ET.fromstring(response.content)
                items = root.findall('.//item')

                data_list = []
                for item in items:
                    prd_nm = item.find('prdNm').text if item.find('prdNm') is not None else ""
                    cn_sumry = item.find('cnSumry').text if item.find('cnSumry') is not None else ""
                    if cn_sumry:
                        data_list.append({"품목/제품": prd_nm, "소비자 불만 및 피해 요약": cn_sumry})

                if data_list:
                    return pd.DataFrame(data_list)
            else:
                print(f"KCA API 호출 실패: 상태 코드 {response.status_code}")
        else:
            print("KCA_API_KEY not set. Skipping KCA API call.")

    except Exception as e:
        print(f"KCA API 호출 오류: {e}")

    backup_data = {
        "품목/제품": [product_name],
        "소비자 불만 및 피해 요약": ["초기 제품 하자, 공식 서비스센터(A/S) 대기 시간 지연 및 수리비 청구 분쟁, 배터리 효율 저하 문제 발생."]
    }
    return pd.DataFrame(backup_data)

# ==========================================
# 3. YouTube 실시간 댓글 수집 함수
# ==========================================
def get_youtube_comments_and_thumbnails(product_name):
    comments = []
    thumbnail_urls = []

    include_keywords = ['좋다', '좋음', '별로', '나쁘다', '만족', '불만족', '장점', '단점', '후회', '추천', '비추', '성능', '배터리', '카메라', '디자인', '속도', '발열', '소리', '화면', '기능', '편리', '불편', '문제', '해결', '고장', '오류', '서비스']
    exclude_keywords = ['영상', '리뷰', '설명', '감사', '구독', '좋아요', '질문', '가격', '어디서', '어떻게', '링크', '구매', '궁금', '문의', '얼마', '해주세요', '사용 시간', '배터리 시간', '궁금해요', '어떤가요', '몇시간', '알고 샀습니다', '만족합니다', '걸려서 샀는데', '후회없어요', '좋네요', '괜찮네요', '별로에요', '최고', '대박', '진짜', '중고', '윈도우']

    try:
        # YOUTUBE_API_KEY가 설정되어 있어야만 API 호출을 시도합니다.
        if YOUTUBE_API_KEY:
            youtube = googleapiclient.discovery.build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

            search_response = youtube.search().list(
                q=f"{product_name} 실제 후기 단점",
                part="id,snippet",
                maxResults=3,
                type="video"
            ).execute()

            video_ids = [item['id']['videoId'] for item in search_response.get('items', [])]

            for item in search_response.get('items', []):
                if 'thumbnails' in item['snippet'] and 'high' in item['snippet']['thumbnails']:
                    thumbnail_urls.append(item['snippet']['thumbnails']['high']['url'])

            for video_id in video_ids:
                try:
                    comment_response = youtube.commentThreads().list(
                        videoId=video_id,
                        part="snippet",
                        maxResults=100
                    ).execute()

                    for item in comment_response.get('items', []):
                        comment = item['snippet']['topLevelComment']['snippet']['textDisplay']
                        cleaned = clean_text(comment)

                        if len(cleaned) <= 10:
                            continue

                        is_evaluative = any(keyword in cleaned for keyword in include_keywords)
                        is_non_evaluative = any(keyword in cleaned for keyword in exclude_keywords)

                        if is_evaluative and not is_non_evaluative:
                            comments.append(cleaned)
                except Exception:
                    continue
            return comments[:500], thumbnail_urls
        else:
            print("YOUTUBE_API_KEY not set. Skipping YouTube API call.")
            return [], []

    except Exception as e:
        print(f"YouTube API 호출 오류: {e}")
        return [], []

# ==========================================
# 4. Gemini AI 자연어 분석 및 요약 함수
# ==========================================
def analyze_product_with_gemini(product_name, comments, kca_df):
    if not model:
        return "Gemini API 키가 설정되지 않아 AI 분석을 수행할 수 없습니다."

    if not comments and kca_df.empty:
        return "분석할 실시간 데이터가 존재하지 않습니다."

    kca_context = kca_df.to_string(index=False) if not kca_df.empty else "소비자원 공식 접수 불만 데이터 없음."
    comments_blob = "\\n".join([f"- {c}" for c in comments[:150]]) if comments else "유튜브 실시간 리뷰 댓글 없음."

    prompt = f"""
    당신은 테크 제품 객관적 구매 가이드를 작성하는 AI 전문가입니다.
    다음 제공된 한국소비자원의 실제 피해 통계자료와 유튜브의 실시간 사용자 댓글들을 정밀 문맥 분석해 주세요.

    [소비자원 공식 접수 불만 데이터]
    {kca_context}

    [유튜브 실시간 리뷰 댓글들]
    {comments_blob}

    유튜브 실시간 사용자 댓글에는 영어 댓글도 포함될 수 있습니다. 영어 댓글은 자연스럽게 한국어로 번역하여 분석에 포함해 주세요.
    광고성 멘트를 완전히 배제하고 소비자가 체감하는 단점과 진솔한 평판 위주로 분석하여 아래 포맷으로만 출력하세요.
    비율 수치는 주어진 텍스트의 감성 문맥을 바탕으로 가장 근접하게 추정하여 %로 제공하세요.
    두 비율(긍정/부정)의 합은 반드시 100%가 되어야 합니다.

    ### 📊 긍정 / 부정 비율 분석
    - 긍정 비율: XX%
    - 부정 비율: XX%

    ### 👍 실제 사용자들이 꼽은 장점 (사야 하는 이유 3가지)
    1.
    2.
    3.

    ### 👎 소비자원 및 댓글 기반 치명적 단점 (거르는 게 좋은 이유 3가지)
    1.
    2.
    3.
    """

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Gemini AI 분석 중 오류 발생: {e}"

def recommend_alternative_products(product_name, disadvantages):
    if not model:
        return "Gemini API 키가 설정되지 않아 대체 제품 추천을 수행할 수 없습니다."

    if not disadvantages:
        return ""

    recommendation_prompt = f"""
    사용자가 다음 제품 '{product_name}'에 대해 다음과 같은 치명적인 단점들을 발견했습니다:
    {disadvantages}

    이러한 단점들을 보완하거나 피할 수 있는 3가지 대체 제품을 추천해 주세요. 각 추천 제품에 대해 간략한 설명과 함께 해당 단점을 어떻게 해결하는지 명시해 주세요.
    포맷은 다음과 같이 해주세요:

    ### 💡 더 나은 대체 제품 후보
    1. [제품 이름]: [간략한 설명 및 단점 보완점]
    2. [제품 이름]: [간략한 설명 및 단점 보완점]
    3. [제품 이름]: [간략한 설명 및 단점 보완점]
    """
    try:
        response = model.generate_content(recommendation_prompt)
        return response.text
    except Exception as e:
        return f"대체 제품 추천 중 오류 발생: {e}"


# ==========================================
# 5. Gradio 웹 인터페이스 화면 구성
# ==========================================
def generate_purchase_guide(product_name):
    if not product_name:
        return "제품명을 입력해주세요."

    # 1. 한국소비자원 공공데이터 수집
    kca_df = get_kca_data(product_name)

    # 2. 유튜브 댓글 수집 및 정제
    comments, thumbnail_urls = get_youtube_comments_and_thumbnails(product_name)

    # 3. Gemini AI 리포트 도출
    report = analyze_product_with_gemini(product_name, comments, kca_df)

    full_report = f"# '{product_name}' 구매 가이드 리포트\\n\\n"
    full_report += "---\\n\\n" # Horizontal rule for main separation

    # Add product images if available
    if thumbnail_urls:
        full_report += "## 📸 제품 관련 이미지\\n\\n"
        for url in thumbnail_urls:
            full_report += f"<img src='{url}' width='200' style='margin-right: 10px; border-radius: 8px;'>\\n"
        full_report += "\\n---\\n\\n"

    full_report += "## 📊 데이터 수집 현황\\n\\n" # New section header
    full_report += "### 소비자원 공식 접수 불만 데이터\\n" + kca_df.to_markdown(index=False) + "\\n\\n"
    full_report += "### 유튜브 실시간 리뷰 댓글 (샘플)\\n" + pd.DataFrame(comments[:5], columns=["필터링 완료된 댓글"]).to_markdown(index=False) + "\\n\\n"
    full_report += "---\\n\\n" # Separator before AI analysis

    full_report += "## 🧠 AI 분석 기반 구매 가이드\\n\\n" # New section header
    full_report += report + "\\n\\n" # Ensure a newline after the report content

    # 4. 긍정/부정 비율 분석 및 대체 제품 추천
    positive_match = re.search(r'- 긍정 비율: (\\d+)%', report)
    negative_match = re.search(r'- 부정 비율: (\\d+)%', report)

    positive_ratio = int(positive_match.group(1)) if positive_match else 0
    negative_ratio = int(negative_match.group(1)) if negative_match else 0

    if negative_ratio > positive_ratio:
        # '👎 소비자원 및 댓글 기반 치명적 단점' 섹션 추출
        disadvantages_match = re.search(r'### 👎 소비자원 및 댓글 기반 치명적 단점 \\\\(거르는 게 좋은 이유 3가지\\\)(.*?)(?:###|\\Z)', report, re.DOTALL)
        disadvantages = disadvantages_match.group(1).strip() if disadvantages_match else ""

        if disadvantages:
            recommendations = recommend_alternative_products(product_name, disadvantages)
            full_report += "---\\n\\n## ✨ 대체 제품 추천\\n\\n" + recommendations # New section header for recommendations
        else:
            full_report += "---\\n\\n## ✨ 대체 제품 추천\\n\\n(단점 분석이 불가능하여 추천이 어렵습니다.)"


    return full_report


iface = gr.Interface(
    fn=generate_purchase_guide,
    inputs=gr.Textbox(lines=1, label="분석하려는 가전/테크 제품명을 정확히 입력하세요. (예: 갤럭시 S24, 맥북 에어 M3)"),
    outputs=gr.Markdown(label="AI 분석 기반 구매 가이드 리포트"),
    title="🔍 AI 기반 테크 제품 구매 가이드",
    description="제품명을 입력하면 한국소비자원 불만 데이터와 유튜브 리뷰 댓글을 분석하여 AI 구매 가이드를 제공합니다.",
    submit_btn="분석 시작",
    clear_btn="초기화",
    flagging_mode='never'
)

# Hugging Face Spaces에 배포할 때는 share=False로 설정해야 합니다.
iface.launch(share=False, debug=False)
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_py_content)

print("app.py 파일이 생성되었습니다. API 키는 Hugging Face Spaces의 Secrets에서 설정해야 합니다.")

In [ ]:
# requirements.txt 파일 생성
# 필요한 모든 라이브러리를 여기에 나열합니다.
requirements_content = """
gradio
google-generativeai
pandas
requests
google-api-python-client
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("requirements.txt 파일이 생성되었습니다.")

### 3. Hugging Face Space로 파일 푸시

이제 `huggingface_hub`를 사용하여 생성한 `app.py`와 `requirements.txt` 파일을 Hugging Face Space 리포지토리에 업로드합니다.

**반드시 아래 코드의 `<your-username>`과 `<your-space-name>`을 본인의 Hugging Face 사용자 이름과 생성한 Space 이름으로 변경하세요.**

(아직 Hugging Face Space를 생성하지 않았다면, [Hugging Face Spaces](https://huggingface.co/spaces)에서 `+ New Space` 버튼을 클릭하여 `Space name` 입력 후 `Space SDK`를 `Gradio`로 선택하여 생성해주세요.)

In [ ]:
from huggingface_hub import HfApi, Repository
import os

# --- 사용자 정보 설정 (반드시 변경하세요!) ---
YOUR_USERNAME = "<your-username>" # 본인의 Hugging Face 사용자 이름 예: 'your-username'
YOUR_SPACE_NAME = "<your-space-name>" # 위에서 생성한 Space 이름 예: 'my-gradio-app'
# --------------------------------------------

# Hugging Face Space의 전체 ID
repo_id = f"{YOUR_USERNAME}/{YOUR_SPACE_NAME}"

# Colab 작업 디렉토리에 'repo' 폴더를 생성하고 해당 Space의 리포지토리를 클론합니다.
repo_dir = "./repo"

# 기존에 'repo' 디렉토리가 있다면 삭제하고 다시 시작합니다.
if os.path.exists(repo_dir):
    !rm -rf {repo_dir}

# 리포지토리를 클론합니다.
# 이미 `login()` 함수로 인증을 완료했으므로, 리포지토리 생성 또는 클론 시 인증 문제가 없습니다.
repo = Repository(local_dir=repo_dir, clone_from=repo_id)

# app.py와 requirements.txt 파일을 클론한 리포지토리 폴더로 이동합니다.
# 파일이 생성되지 않았다면 오류가 발생할 수 있으므로, 이전 셀들을 먼저 실행해야 합니다.
if os.path.exists("app.py"):
    !mv app.py {repo_dir}/
else:
    print("Warning: app.py not found. Please run the previous cell to create it.")

if os.path.exists("requirements.txt"):
    !mv requirements.txt {repo_dir}/
else:
    print("Warning: requirements.txt not found. Please run the previous cell to create it.")

# 변경사항을 커밋하고 푸시합니다.
# 'repo.git_add()', 'repo.git_commit()', 'repo.git_push()' 대신 'with repo.commit():' 사용
with repo.commit("Add Gradio app.py and requirements.txt"): # 커밋 메시지
    print("Files committed locally.")

print(f"Pushing files to Hugging Face Space: {repo_id}")
# git push를 명시적으로 호출
repo.git_push()

print(f"Files pushed to Hugging Face Space: {repo_id}")
print(f"생성된 Space는 다음 링크에서 확인할 수 있습니다: https://huggingface.co/spaces/{repo_id}")

### 4. Hugging Face Space Secrets 설정 (수동 작업)

`app.py`에서 `os.getenv()`를 사용하여 API 키를 불러오도록 변경했으므로, 이 키들은 Hugging Face Spaces 웹사이트에서 수동으로 설정해야 합니다.

1.  Hugging Face 웹사이트에서 본인의 Space 페이지로 이동합니다.
2.  `Settings` 탭을 클릭합니다.
3.  좌측 메뉴에서 `Repository secrets`를 찾아 클릭합니다.
4.  `New secret` 버튼을 클릭하여 다음 키-값 쌍을 추가합니다:
    *   `KCA_API_KEY`: 본인의 한국소비자원 API 키
    *   `YOUTUBE_API_KEY`: 본인의 YouTube API 키
    *   `GEMINI_API_KEY`: 본인의 Gemini API 키
5.  모든 키를 추가하면 Space가 자동으로 재시작되며, 앱이 정상적으로 배포될 것입니다.